In [2]:
import mediapipe as mp

In [7]:
import cv2
import mediapipe as mp

mp_face_detection = mp.solutions.face_detection
mp_drawing = mp.solutions.drawing_utils

cap = cv2.VideoCapture('threeman_talk.mp4')
with mp_face_detection.FaceDetection(
    model_selection=0, min_detection_confidence=0.7) as face_detection:
  while cap.isOpened():
    success, image = cap.read()
    if not success:
      break

    image.flags.writeable = False
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    results = face_detection.process(image)
    # print('이미지 : ', image.shape)

    image.flags.writeable = True
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

    if results.detections:
        # 6개의 위치 : 오른쪽 눈, 왼쪽 눈, 코 끝, 입 중심, 오른쪽 이주, 왼쪽 이주
      for detection in results.detections:
        # mp_drawing.draw_detection(image, detection)
        # print(detection)

        keypoints = detection.location_data.relative_keypoints
        right_eye = keypoints[0]   # 오른쪽 눈의 비율값
        left_eye = keypoints[1]
        nose_tip = keypoints[2]
        # print(right_eye)

        h, w, _ = image.shape
        right_eye = (int(right_eye.x * w)  , int( right_eye.y * h) )
        left_eye = (int(left_eye.x * w)   , int( left_eye.y * h))
        nose_tip = (int(nose_tip.x * w)  , int( nose_tip.y * h))
        # print(right_eye)

        # 양쪽 눈에 원 그리기
        cv2.circle(image, right_eye, 30, (255, 0, 0), 10, cv2.LINE_AA)
        cv2.circle(image, left_eye, 30, (255, 0, 0), 10, cv2.LINE_AA)
        cv2.circle(image, nose_tip, 40, (255, 0, 0), 10, cv2.LINE_AA)

    cv2.imshow('MediaPipe Face Detection', image)
    if cv2.waitKey(1) == ord('q'):
      break
cap.release()
cv2.destroyAllWindows()

In [59]:
import cv2
import mediapipe as mp

mp_face_detection = mp.solutions.face_detection
mp_drawing = mp.solutions.drawing_utils

image_right_eye = cv2.imread('right_eye.png')
image_left_eye = cv2.imread('left_eye.png')
image_nose = cv2.imread('nose.png')

cap = cv2.VideoCapture('treeman_talk.mp4')
with mp_face_detection.FaceDetection(
    model_selection=0, min_detection_confidence=0.7) as face_detection:
  while cap.isOpened():
    success, image = cap.read()
    if not success:
      break

    image.flags.writeable = False
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    results = face_detection.process(image)
    # print('이미지 : ', image.shape)

    image.flags.writeable = True
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    if results.detections:
        # 6개의 위치 : 오른쪽 눈, 왼쪽 눈, 코 끝, 입 중심, 오른쪽 이주, 왼쪽 이주
      for detection in results.detections:
        # mp_drawing.draw_detection(image, detection)
        # print(detection)

        keypoints = detection.location_data.relative_keypoints
        right_eye = keypoints[0]
        left_eye = keypoints[1]
        nose_tip = keypoints[2]

        h, w, _ = image.shape
        right_eye = (int(right_eye.x * w) -10  , int( right_eye.y * h)-70 )
        left_eye = (int(left_eye.x * w) +10  , int( left_eye.y * h)-70)
        nose_tip = (int(nose_tip.x * w)  , int( nose_tip.y * h)+30)

        image[right_eye[1]-50:right_eye[1]+50 , right_eye[0]-50:right_eye[0]+50] =image_right_eye 
        image[left_eye[1]-50:left_eye[1]+50 , left_eye[0]-50:left_eye[0]+50] =image_left_eye 
        image[nose_tip[1]-50:nose_tip[1]+50 , nose_tip[0]-100:nose_tip[0]+100] =image_nose 

    cv2.imshow('MediaPipe Face Detection', image)
    if cv2.waitKey(1) == ord('q'):
      break
cap.release()
cv2.destroyAllWindows()

In [10]:
cv2.imread('right_eye.png', cv2.IMREAD_UNCHANGED).shape

(100, 100, 4)

In [60]:
import cv2
import mediapipe as mp
import numpy as np

mp_face_detection = mp.solutions.face_detection
# mp_drawing = mp.solutions.drawing_utils

image_right_eye = cv2.imread('right_eye.png', cv2.IMREAD_UNCHANGED)
image_left_eye = cv2.imread('left_eye.png', cv2.IMREAD_UNCHANGED)
image_nose = cv2.imread('nose.png', cv2.IMREAD_UNCHANGED)

def overlay(image, x, y, w, h, over_image):
    alpha = over_image[:, :, 3]  # 채널 BGRA
    mask_image = alpha / 255.0
    for c in range(3):
        image[y-h:y+h ,x-w:x+w, c] = (over_image[:, :, c] * mask_image) +  (image[y-h:y+h ,x-w:x+w, c] * (1-mask_image))

cap = cv2.VideoCapture('treeman_talk.mp4')

with mp_face_detection.FaceDetection(model_selection=0, min_detection_confidence=0.7) as face_detection:
    while cap.isOpened():
        success, image = cap.read()         
        if not success: break 
        # image.flags.writeable = False
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = face_detection.process(image)
        # print('이미지 : ', image.shape)
        
        # Draw the face detection annotations on the image.
        # image.flags.writeable = True
        # image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
        if results.detections:
            # 6개의 위치 : 오른쪽 눈, 왼쪽 눈, 코 끝, 입 중심, 오른쪽 이주, 왼쪽 이주
            detection = results.detections[0]
            # mp_drawing.draw_detection(image, detection)
            # print(detection)           
            keypoints = detection.location_data.relative_keypoints
            right_eye = keypoints[0]
            left_eye = keypoints[1]
            nose_tip = keypoints[2]
            
            h, w, _ = image.shape
            right_eye = (int(right_eye.x * w) -10, int( right_eye.y * h)-70 )
            left_eye = (int(left_eye.x * w) +10, int( left_eye.y * h)-70)
            nose_tip = (int(nose_tip.x * w), int( nose_tip.y * h)+30)
            
            # image, x, y, w, h, over_image
            overlay(image, *right_eye, 50, 50, image_right_eye )
            overlay(image, *left_eye, 50, 50, image_left_eye )
            overlay(image, *nose_tip, 100, 50, image_nose )
              
        # Flip the image horizontally for a selfie-view display.
        cv2.imshow('MediaPipe Face Detection', image)
        if cv2.waitKey(1) == ord('q'):  break
cap.release()
cv2.destroyAllWindows()

In [15]:
import cv2
nose = cv2.imread('right_eye.png', cv2.IMREAD_UNCHANGED)
nose.shape

(100, 100, 4)

In [16]:
import cv2
nose = cv2.imread('left_eye.png', cv2.IMREAD_UNCHANGED)
nose.shape

(100, 100, 4)

In [14]:
import cv2
cv2.imread('nose.png').shape

(100, 200, 3)

In [22]:
import cv2
import numpy as np

# 이미지 읽기 (BGR)
img = cv2.imread('left_eye.png')

# BGRA로 변환 (알파 채널 추가)
img = cv2.cvtColor(img, cv2.COLOR_BGR2BGRA)

# 흰색 기준 설정
lower = np.array([240, 240, 240, 255])
upper = np.array([255, 255, 255, 255])

# 마스크 생성
mask = cv2.inRange(img, lower, upper)

# 흰색 부분 → 투명(alpha=0)
img[mask > 0] = [0, 0, 0, 0]

# 저장
cv2.imwrite('left_eye2.png', img)

True

In [61]:
import cv2
import mediapipe as mp
import numpy as np

# 이미지 로드 (파일 경로 확인 필수!)
image_right_eye = cv2.imread('right_eye.png', cv2.IMREAD_UNCHANGED)
image_left_eye = cv2.imread('left_eye.png', cv2.IMREAD_UNCHANGED)
image_nose = cv2.imread('nose.png', cv2.IMREAD_UNCHANGED)

def overlay(image, x, y, w_size, h_size, over_image):
    if over_image is None: return
    
    # 1. 원하는 크기(w_size, h_size)로 합성 이미지 리사이즈 (중요!)
    # w_size, h_size는 중심으로부터의 거리이므로 전체 폭은 * 2
    target_w = w_size * 2
    target_h = h_size * 2
    over_image = cv2.resize(over_image, (target_w, target_h))
    
    # 2. 합성 영역 좌표 계산
    y1, y2 = y - h_size, y + h_size
    x1, x2 = x - w_size, x + w_size
    
    # 3. 화면 밖으로 나가는 경우 예외 처리 (팅김 방지)
    img_h, img_w = image.shape[:2]
    if x1 < 0 or y1 < 0 or x2 > img_w or y2 > img_h:
        return

    # 4. 알파 채널 분리 및 정규화 (0~1)
    alpha = over_image[:, :, 3] / 255.0
    mask_image = alpha
    
    # 5. 합성 (브로드캐스팅 활용으로 속도 향상)
    for c in range(3):
        # (합성 이미지 * 투명도) + (배경 이미지 * (1 - 투명도))
        image[y1:y2, x1:x2, c] = (over_image[:, :, c] * mask_image + 
                                  image[y1:y2, x1:x2, c] * (1 - mask_image)).astype(np.uint8)

# --- 메인 루프 ---
mp_face_detection = mp.solutions.face_detection
cap = cv2.VideoCapture('treeman_talk.mp4')

with mp_face_detection.FaceDetection(model_selection=0, min_detection_confidence=0.7) as face_detection:
    while cap.isOpened():
        success, image = cap.read()
        if not success: break

        # 처리 속도를 위해 가급적 필수는 아니지만 RGB 변환
        results = face_detection.process(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))

        if results.detections:
            detection = results.detections[0]
            keypoints = detection.location_data.relative_keypoints
            h, w, _ = image.shape
            
            # 좌표 계산 (비율값 * 픽셀 크기)
            # 오프셋(-70, +30 등)은 사용자의 영상에 맞춰 조절
            r_eye_x, r_eye_y = int(keypoints[0].x * w) - 10, int(keypoints[0].y * h) - 70
            l_eye_x, l_eye_y = int(keypoints[1].x * w) + 10, int(keypoints[1].y * h) - 70
            nose_x, nose_y = int(keypoints[2].x * w), int(keypoints[2].y * h) + 30

            # 합성 실행
            overlay(image, r_eye_x, r_eye_y, 50, 50, image_right_eye)
            overlay(image, l_eye_x, l_eye_y, 50, 50, image_left_eye)
            overlay(image, nose_x, nose_y, 100, 50, image_nose)

        cv2.imshow('MediaPipe Face Detection', image)
        if cv2.waitKey(50) == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()